In [34]:
#data builder
#build csv list of JB HI-FIs in NSW with google maps api

import requests

# import api keys from .env file
import os

GOOGLE_MAPS_API_KEY=os.getenv("GOOGLE_MAPS_API_KEY")
GOOGLE_MAPS_DEMO_API_KEY=os.getenv("GOOGLE_MAPS_DEMO_API_KEY")

response = requests.post(
    "https://places.googleapis.com/v1/places:searchText",
    headers={
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_DEMO_API_KEY,
        "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.priceLevel",
    },
    json={
        "textQuery": "JB Hi-Fi in New South Wales, Australia"
    },
)

# show response
response.json()

{'places': [{'formattedAddress': 'Homebush Direct Factory Outlet, Store 2 - 007/3-5 Underwood Rd, Homebush NSW 2140, Australia',
   'displayName': {'text': 'JB Hi-Fi Homebush HOME', 'languageCode': 'en'}},
  {'formattedAddress': 'Westfield Sydney Shopping Centre Store 6010, Level 6 Corner Pitt &, Market St, Sydney NSW 2000, Australia',
   'displayName': {'text': 'JB Hi-Fi City - Westfield Sydney',
    'languageCode': 'en'}},
  {'formattedAddress': 'World Square Shopping Centre Store 9.42, Lower Ground Level, 644 George St, Sydney NSW 2000, Australia',
   'displayName': {'text': 'JB Hi-Fi City - World Square',
    'languageCode': 'en'}},
  {'formattedAddress': 'The Broadway Shopping Centre, Store 207, Level 2/1 Bay St, Ultimo NSW 2007, Australia',
   'displayName': {'text': 'JB Hi-Fi Broadway', 'languageCode': 'en'}},
  {'formattedAddress': 'Top Ryde Shopping Centre Store 29, Lower Ground 1 Level Corner Devlin Street &, Blaxland Rd, Ryde NSW 2112, Australia',
   'displayName': {'text': 

In [35]:

depot_location = {
    "Name": "JB Hi-Fi Sydney Home Delivery Centre",
    "Address": "Bldg 6/145 Hollinsworth Rd, Marsden Park NSW 2765, Australia",
    "Demand": 0,
    "ReadyTime": 0,
    "DueDate": 480,
    "ServiceTime": 0
}
# for every element in response.json()['places'], create a new csv, add to row in csv with displayName, formattedAddresswith headings Name, Address

# add realistic demand, ready time, due date, service time for each location in the csv for vehicle routing problem with time windows (VRPTW) optimization. assume the time is all in minutes so consider that a full time day work is 9 to 5, 8 hours of work so 8*60 is 480 minutes. time window should be between 0 and 480 minutes. ready time should be between 0 and 480 minutes. due date should be between 0 and 480 minutes. service time should be between 0 and 60 minutes. demand should be between 1 and 10 units. add these columns to the csv with headings Demand, ReadyTime, DueDate, ServiceTime
import random

#set random seed for reproducibility
random.seed(42)
# but address includes commas, so we need to wrap the address in quotes in the csv file.
csv_file = 'jb_hifi_nsw.csv'
with open(csv_file, 'w') as f:
    f.write('Name,Address,Demand,ReadyTime,DueDate,ServiceTime\n')
    f.write(f"\"{depot_location['Name']}\",\"{depot_location['Address']}\",{depot_location['Demand']},{depot_location['ReadyTime']},{depot_location['DueDate']},{depot_location['ServiceTime']}\n")
    for place in response.json()['places']:
        demand = random.randint(1, 10)
        ready_time = random.randint(0, 480)
        due_date = random.randint(0, 480)
        service_time = random.randint(0, 60)
        f.write(f"\"{place['displayName']['text']}\",\"{place['formattedAddress']}\",{demand},{ready_time},{due_date},{service_time}\n")

In [32]:
# write a python function that takes in a list of locations and returns the distance matrix using the google maps api. The function should take in a list of locations, where each location is a dictionary with keys 'Name' and 'Address'. The function should return a 2D list where the element at index [i][j] is the distance in meters from location i to location j. The function should use the google maps api to get the distance between each pair of locations. The function should also handle any errors that may occur when calling the google maps api.

response = requests.post(
    "https://routes.googleapis.com/directions/v2:computeRoutes",
    headers={
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": "routes.duration,routes.distanceMeters,routes.routeToken,routes.travelAdvisory.routeRestrictionsPartiallyIgnored",
    },
    json={
        "origin": {
            "address": "Bldg 6/145 Hollinsworth Rd, Marsden Park NSW 2765, Australia"
        },
        "destination": {
            "address": "Level 2/500 Oxford St, Bondi Junction NSW 2022, Australia"
        },
        "travelMode": "TRUCK",
        "routingPreference": "TRAFFIC_AWARE_OPTIMAL",
    },
)

response.json()

{'routes': [{'distanceMeters': 53468,
   'duration': '2666s',
   'travelAdvisory': {},
   'routeToken': 'CrYCCsUBMsIBGqYBCjcCFg5XkDSOic4Fz02RjwKpqCfSng_k_0XkuojdAq-14d0C8bAG6o0I9YujAZuRIO7FhwGN9QEAEjQp-SZemRgMerxRBgYew1G3Ly4eq6oopnpMUV_pdnSkUPjhA7N0oNMmTzWY-gqx4xd6jDD1Gg4AHykvCkHnPwEFWyU1ACoNWg4LBiBdEW8AbjATBTIBCD1vMKc-RQAAgD9I5JqynLvSy4S6ASIXNUpiRGF2NkVFNVhNcjd3UHZkakpzQU0QBRpTClEKGAoNCgIIAREAAAAAAIBmQBFU46Wb5Km5QBIWCAAQAxAGEBMQEhgCQgQaAggFSgIIASIbChc1SmJEYXNIY0VaWE1yN3dQdmRqSnNBTXABKAEiFQB5Tdj410xMzn0PIsOyCjZrejt6sw'}]}